In [2]:
import os
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.efficientnet import preprocess_input
from tqdm import tqdm
from collections import defaultdict

# --- Configuration ---
model_path = "/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/code/data/snapshots/all/hypVSadn_HDall2023_efficientnet_0_regularized0.0_256x256_1in_nf64_bnTrue_fcdo0.0_balancedTrue_loss_fl_gamma1.0_sgd_5fold0_best.h5"
test_file   = "/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/imagesets_characterisation/test_update.txt"
image_size  = (256, 256)
top_k_per_class = 10

# --- Load model (compile=False) ---
model = load_model(model_path, compile=False)

# --- Helper function: preprocess a single image ---
def preprocess_image(image_path):
    image = load_img(image_path, target_size=image_size)
    image_array = img_to_array(image)
    image_array = preprocess_input(image_array)
    return image_array

# --- Two dictionaries to store confidence scores per sequence, one for each class ---
seq_scores_class0 = defaultdict(list)
seq_scores_class1 = defaultdict(list)

# --- Read test file ---
with open(test_file, 'r') as f:
    lines = f.readlines()

# --- Process each image listed in the test file ---
for line in tqdm(lines, desc="Processing test images"):
    img_path, ann_path, label_str, seq_id = line.strip().split()
    label = int(label_str)

    # Skip if image file doesn't exist
    if not os.path.exists(img_path):
        continue

    # Determine sequence name (folder name containing the image)
    sequence_name = os.path.basename(os.path.dirname(img_path))

    # Load and preprocess the image
    image = preprocess_image(img_path)
    image = np.expand_dims(image, axis=0)  # Add batch dimension

    # Predict with the model
    prediction = model.predict(image, verbose=0)
    confidence = float(np.max(prediction))  # Max softmax score

    # Store confidence in the appropriate dict based on label
    if label == 0:
        seq_scores_class0[sequence_name].append(confidence)
    else:
        seq_scores_class1[sequence_name].append(confidence)

# --- Compute average confidence per sequence for each class ---
avg_scores_class0 = []
for seq, scores in seq_scores_class0.items():
    if len(scores) > 0:
        avg_scores_class0.append((seq, np.mean(scores)))

avg_scores_class1 = []
for seq, scores in seq_scores_class1.items():
    if len(scores) > 0:
        avg_scores_class1.append((seq, np.mean(scores)))

# --- Sort sequences by descending average confidence ---
avg_scores_class0.sort(key=lambda x: x[1], reverse=True)
avg_scores_class1.sort(key=lambda x: x[1], reverse=True)

# --- Display Top K sequences for each class ---
print(f"\nTop {top_k_per_class} sequences for class 0 (label 0):")
for i, (seq_name, score) in enumerate(avg_scores_class0[:top_k_per_class], 1):
    print(f"{i}. {seq_name}: {score:.4f}")

print(f"\nTop {top_k_per_class} sequences for class 1 (label 1):")
for i, (seq_name, score) in enumerate(avg_scores_class1[:top_k_per_class], 1):
    print(f"{i}. {seq_name}: {score:.4f}")


2025-06-04 13:11:32.070752: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublas.so.11'; dlerror: libcublas.so.11: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda/extras/CUPTI/lib64:/usr/local/cuda/compat/lib:/usr/local/nvidia/lib:/usr/local/nvidia/lib64:/.singularity.d/libs
2025-06-04 13:11:32.071880: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublasLt.so.11'; dlerror: libcublasLt.so.11: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda/extras/CUPTI/lib64:/usr/local/cuda/compat/lib:/usr/local/nvidia/lib:/usr/local/nvidia/lib64:/.singularity.d/libs
2025-06-04 13:11:32.073082: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcufft.so.10'; dlerror: libcufft.so.10: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/loc


Top 10 sequences for class 0 (label 0):
1. p451_05: 0.7467
2. p730_02_01: 0.7247
3. p807_02_02: 0.7228
4. p796_03_03: 0.7148
5. p046_19_R_WLI: 0.7131
6. p525_13: 0.7068
7. p228_01_R_LCI: 0.6905
8. p526_03: 0.6903
9. p526_02: 0.6901
10. p525_08: 0.6899

Top 10 sequences for class 1 (label 1):
1. p364_04: 0.8929
2. p415_01: 0.8713
3. p028_09_R_WLI: 0.8297
4. p584_07: 0.8252
5. p457_01: 0.8227
6. p028_08_R_LCI: 0.8201
7. p643_1_01: 0.8066
8. p295_02_R_BLI: 0.7991
9. p047_04_R_WLI: 0.7886
10. p871_02_01: 0.7764


In [3]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# ------------------------------------------------------------------
# Lists of sequences as provided by the user:
# 10 for class 0 and 10 for class 1.
# ------------------------------------------------------------------
class0_sequences = [
    "p451_05", "p730_02_01", "p807_02_02", "p796_03_03", "p046_19_R_WLI",
    "p525_13", "p228_01_R_LCI", "p526_03", "p526_02", "p525_08"
]

class1_sequences = [
    "p364_04", "p415_01", "p028_09_R_WLI", "p584_07", "p457_01",
    "p028_08_R_LCI", "p643_1_01", "p295_02_R_BLI", "p047_04_R_WLI", "p871_02_01"
]

top_sequences = class0_sequences + class1_sequences

# ------------------------------------------------------------------
# Base directories (adjust if your folder structure is different)
# ------------------------------------------------------------------
sequences_root = "/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/outputs_crops/256p_contrast_enhanced"
output_root    = "/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/outputs_crops/256p_artif_degraded"

# ------------------------------------------------------------------
# Overexposure (brightening) function – supports 'global' or 'gaussian' modes
# ------------------------------------------------------------------
def overlight_image(img, mode='global', intensity=1.5):
    """
    If mode == 'global':
        multiplies all pixel values by 'intensity' (clamped to [0,255]).
    If mode == 'gaussian':
        creates a 2D Gaussian mask, normalizes it to [0,1],
        and adds (intensity * 255 * mask) to the original image.
    """
    img = img.astype(np.float32)

    if mode == 'global':
        result = img * intensity
        return np.clip(result, 0, 255).astype(np.uint8)

    elif mode == 'gaussian':
        h, w = img.shape[:2]
        # Create 1D Gaussian kernels for width and height
        x = cv2.getGaussianKernel(w, w // 3)
        y = cv2.getGaussianKernel(h, h // 3)
        gaussian = y @ x.T
        gaussian = (gaussian - gaussian.min()) / (gaussian.max() - gaussian.min())
        mask = np.stack([gaussian] * 3, axis=-1)  # convert to 3 channels

        # Add controlled brightening by intensity
        result = img + intensity * 255.0 * mask
        return np.clip(result, 0, 255).astype(np.uint8)

    else:
        raise ValueError(f"Unknown mode '{mode}'. Use 'global' or 'gaussian'.")

# ------------------------------------------------------------------
# Gaussian blur function – kernel size is based on blur_factor (must be odd)
# ------------------------------------------------------------------
def blur_image(img, blur_factor=5):
    """
    Applies Gaussian blur with a kernel size equal to blur_factor (odd).
    """
    # Ensure blur_factor is odd; if even, add 1
    ksize = blur_factor if (blur_factor % 2 == 1) else (blur_factor + 1)
    return cv2.GaussianBlur(img, (ksize, ksize), 0)

# ------------------------------------------------------------------
# Simple obstruction function (draws a black circle in the center)
# ------------------------------------------------------------------
def add_obstruction(img, shape='circle', size=50):
    h, w = img.shape[:2]
    img_copy = img.copy()

    if shape == 'circle':
        center = (w // 2, h // 2)
        cv2.circle(img_copy, center, size, (0, 0, 0), -1)
    else:
        # You can expand to rectangle, polygon, etc., if needed
        pass

    return img_copy

# ------------------------------------------------------------------
# Degradation parameters:
# - bright_gaussian_intensities: list of smoother intensities (e.g., 0.5, 1.0, 1.5)
# - blur_factors: list of smaller blur factors (e.g., 5, 10, 15)
# ------------------------------------------------------------------
bright_gaussian_intensities = [0.5, 1.0, 1.5]
blur_factors = [5, 10, 15]

# ------------------------------------------------------------------
# Main loop: for each sequence, process every frame
# ------------------------------------------------------------------
for seq in tqdm(top_sequences, desc="Processing sequences"):
    seq_path = os.path.join(sequences_root, seq)
    out_seq_path = os.path.join(output_root, seq)
    os.makedirs(out_seq_path, exist_ok=True)

    # Check if the sequence directory exists
    if not os.path.isdir(seq_path):
        print(f"[!] Directory not found: {seq_path}")
        continue

    for frame_file in os.listdir(seq_path):
        # Skip files that are not images
        if not frame_file.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue

        frame_path = os.path.join(seq_path, frame_file)
        img = cv2.imread(frame_path)
        if img is None:
            print(f"[!] Failed to load image: {frame_path}")
            continue

        base_name = os.path.splitext(frame_file)[0]

        # 1) Save a copy of the original image (no degradation)
        cv2.imwrite(
            os.path.join(out_seq_path, f"{base_name}_original.png"),
            img
        )

        # 2) Global brightening (fixed intensity = 1.5)
        bright_global = overlight_image(img, mode='global', intensity=1.5)
        cv2.imwrite(
            os.path.join(out_seq_path, f"{base_name}_brightGlobal.png"),
            bright_global
        )

        # 3) Gaussian brightening at multiple smoother intensities
        for intensity in bright_gaussian_intensities:
            bright_gauss = overlight_image(img, mode='gaussian', intensity=intensity)
            # Example output name: frame_brightGaussian_0.5.png
            cv2.imwrite(
                os.path.join(out_seq_path, f"{base_name}_brightGaussian_{intensity:.1f}.png"),
                bright_gauss
            )

        # 4) Smaller Gaussian blurs (factors 5, 10, 15)
        for blur in blur_factors:
            blurred = blur_image(img, blur_factor=blur)
            cv2.imwrite(
                os.path.join(out_seq_path, f"{base_name}_blur{blur}.png"),
                blurred
            )

        # 5) Central obstruction (circle)
        obstructed = add_obstruction(img, shape='circle', size=50)
        cv2.imwrite(
            os.path.join(out_seq_path, f"{base_name}_obstructed.png"),
            obstructed
        )


Processing sequences: 100%|████████████████████████████████████████████████████████████████████| 20/20 [01:10<00:00,  3.54s/it]


In [2]:
import os

# ------------------------------------------------------------------
# Configuration: adjust these paths as needed
# ------------------------------------------------------------------

# Path to the root of degraded images (each subfolder is a sequence)
degraded_root = "/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/outputs_crops/256p_artif_degraded"

# Path to the root of annotations (each subfolder is a sequence, files like 00050.png)
annotations_root = "/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/outputs_crops/256p_annotations"

# Path to the imagesets_characterisation file, which contains:
# <original_image_path> <original_annotation_path> <label> <numeric_param>
characterisation_file = "/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/imagesets_characterisation/test.txt"

# Output TXT file where each line will have:
# <degraded_image_path> <annotation_path> <label> <numeric_param>
output_list_file = "/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/degraded_image_list.txt"


# ------------------------------------------------------------------
# Step 1: Parse imagesets_characterisation, build a lookup dict
# Key: (sequence_name, frame_basename) -> (label, numeric_param)
# ------------------------------------------------------------------
lookup = {}

with open(characterisation_file, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue  # skip empty lines

        # Each line is expected to have 4 columns separated by whitespace:
        # 1) original JPEG path, e.g. /.../JPEGImages/1080p/p013_02_B_WLI/00097.png
        # 2) original annotation path, e.g. /.../Annotations/480p/p013_02_B_WLI/00097.png
        # 3) label (e.g. 0 or 1)
        # 4) numeric parameter (e.g. 22)
        parts = line.split()
        if len(parts) < 4:
            # If the line doesn't have at least 4 columns, skip or raise an error
            print(f"[!] Skipping malformed line: {line}")
            continue

        original_img_path = parts[0]
        # parts[1] is the original annotation path (not used directly for our new annotation path)
        label = parts[2]
        numeric_param = parts[3]

        # Extract sequence name and frame basename from the original image path
        # e.g. "/.../JPEGImages/1080p/p013_02_B_WLI/00097.png"
        seq_name = os.path.basename(os.path.dirname(original_img_path))  # "p013_02_B_WLI"
        frame_basename = os.path.splitext(os.path.basename(original_img_path))[0]  # "00097"

        # Store label and numeric_param in the lookup using (sequence, frame) as key
        lookup[(seq_name, frame_basename)] = (label, numeric_param)


# ------------------------------------------------------------------
# Step 2: Iterate over degraded_root, collect lines for each degraded image
# Skip any file containing "brightGaussian_1.0.png" or "brightGaussian_1.5.png"
# ------------------------------------------------------------------

with open(output_list_file, "w") as out_f:
    # Walk through each sequence directory inside degraded_root
    for seq_name in sorted(os.listdir(degraded_root)):
        seq_dir = os.path.join(degraded_root, seq_name)
        if not os.path.isdir(seq_dir):
            continue  # skip non-directory files

        # For each degraded image file in this sequence folder
        for filename in sorted(os.listdir(seq_dir)):
            # We expect names like "00050_blur30.png", "00050_brightGaussian_0.5.png", etc.
            if not filename.lower().endswith((".png", ".jpg", ".jpeg")):
                continue  # skip non-image files

            # Skip files with "brightGaussian_1.0.png" or "brightGaussian_1.5.png"
            if "brightGaussian_1.0.png" in filename or "brightGaussian_1.5.png" in filename:
                continue

            # Extract the frame basename before the first underscore
            # e.g. "00050_blur30.png" -> "00050"
            frame_basename = filename.split("_", 1)[0]

            # Look up the (label, numeric_param) using (seq_name, frame_basename)
            key = (seq_name, frame_basename)
            if key not in lookup:
                print(f"[!] Warning: no entry in characterisation for {key}")
                continue

            label, numeric_param = lookup[key]

            # Construct full paths:
            degraded_path = os.path.join(seq_dir, filename)
            annotation_path = os.path.join(annotations_root, seq_name, f"{frame_basename}.png")

            # Write a line to the output file:
            # <degraded_image_full_path> <annotation_full_path> <label> <numeric_param>
            out_f.write(f"{degraded_path} {annotation_path} {label} {numeric_param}\n")

print(f"Finished writing list to {output_list_file}")


Finished writing list to /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/degraded_image_list.txt


In [2]:
#new dataset
#!/usr/bin/env python3
"""
create_artificial_variants.py

Read a CSV of test images, apply a series of artificial degradations,
and write out the results to disk, preserving names.
"""

import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

# ------------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------------
CSV_TEST     = '/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/test.csv'
OUTPUT_ROOT  = '/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded'
# Make sure the output directory exists
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# Degradation parameters
BRIGHT_GLOBAL_INTENSITY      = 1.5
BRIGHT_GAUSSIAN_INTENSITIES  = [0.5, 1.0, 1.5]
BLUR_FACTORS                 = [5, 10, 15]
OBSTRUCTION_SIZE             = 50  # radius in pixels

# ------------------------------------------------------------------
# 2. Degradation functions
# ------------------------------------------------------------------
def overlight_image(img, mode='global', intensity=1.5):
    img = img.astype(np.float32)
    if mode == 'global':
        result = img * intensity
        return np.clip(result, 0, 255).astype(np.uint8)
    elif mode == 'gaussian':
        h, w = img.shape[:2]
        x = cv2.getGaussianKernel(w, w // 3)
        y = cv2.getGaussianKernel(h, h // 3)
        gaussian = y @ x.T
        gaussian = (gaussian - gaussian.min()) / (gaussian.max() - gaussian.min())
        mask = np.stack([gaussian]*3, axis=-1)
        result = img + intensity * 255.0 * mask
        return np.clip(result, 0, 255).astype(np.uint8)
    else:
        raise ValueError(f"Unknown mode '{mode}'. Use 'global' or 'gaussian'.")

def blur_image(img, blur_factor=5):
    ksize = blur_factor if (blur_factor % 2 == 1) else (blur_factor + 1)
    return cv2.GaussianBlur(img, (ksize, ksize), 0)

def add_obstruction(img, size=50):
    h, w = img.shape[:2]
    img_copy = img.copy()
    center = (w//2, h//2)
    cv2.circle(img_copy, center, size, (0,0,0), -1)
    return img_copy

# ------------------------------------------------------------------
# 3. Load CSV and process each image
# ------------------------------------------------------------------
df = pd.read_csv(CSV_TEST)

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing test images"):
    filename   = row['filename']
    image_path = row['image_path']

    # Attempt to load
    img = cv2.imread(image_path)
    if img is None:
        print(f"[!] Failed to load: {image_path}")
        continue

    # Base output name (without extension)
    base = os.path.splitext(filename)[0]

    # 1) Original copy
    out_path = os.path.join(OUTPUT_ROOT, f"{base}_orig.png")
    cv2.imwrite(out_path, img)

    # 2) Global brightening
    bg = overlight_image(img, mode='global', intensity=BRIGHT_GLOBAL_INTENSITY)
    out_path = os.path.join(OUTPUT_ROOT, f"{base}_brightGlobal.png")
    cv2.imwrite(out_path, bg)

    # 3) Gaussian brightening (various intensities)
    for intensity in BRIGHT_GAUSSIAN_INTENSITIES:
        img_g = overlight_image(img, mode='gaussian', intensity=intensity)
        out_path = os.path.join(
            OUTPUT_ROOT,
            f"{base}_brightGaussian_{int(intensity*10):02d}.png"
        )
        cv2.imwrite(out_path, img_g)

    # 4) Gaussian blur (various factors)
    for factor in BLUR_FACTORS:
        img_b = blur_image(img, blur_factor=factor)
        out_path = os.path.join(OUTPUT_ROOT, f"{base}_blur{factor}.png")
        cv2.imwrite(out_path, img_b)

    # 5) Central obstruction
    img_o = add_obstruction(img, size=OBSTRUCTION_SIZE)
    out_path = os.path.join(OUTPUT_ROOT, f"{base}_obstructed.png")
    cv2.imwrite(out_path, img_o)

print("All images processed and saved under:", OUTPUT_ROOT)


Processing test images: 100%|██████████████████████████████████████████████████████████████| 1470/1470 [03:20<00:00,  7.32it/s]

All images processed and saved under: /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded


In [4]:
import os
import pandas as pd

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------
degraded_root = "/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded"
csv_path      = "/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/test.csv"
output_list   = "/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_degraded_list.txt"

# ------------------------------------------------------------------
# Step 1: Read CSV and build lookup
# ------------------------------------------------------------------
df = pd.read_csv(csv_path)

# Map each base filename (without extension) → (mask_path, label)
lookup = {
    os.path.splitext(row["filename"])[0]: (row["mask_path"], row["label"])
    for _, row in df.iterrows()
}

# For faster lookups, collect the keys in a list
all_keys = list(lookup.keys())

# ------------------------------------------------------------------
# Step 2: Iterate degraded images, match against CSV keys, write output
# ------------------------------------------------------------------
with open(output_list, "w") as out_f:
    for fname in sorted(os.listdir(degraded_root)):
        # skip non-image files
        if not fname.lower().endswith((".png", ".jpg", ".jpeg")):
            continue

        # strip extension
        name_noext = os.path.splitext(fname)[0]

        # find which CSV key this degraded file corresponds to
        match_key = next((k for k in all_keys if name_noext.startswith(k + "_")), None)
        # also allow perfect match (if someone named a file exactly)
        if match_key is None and name_noext in lookup:
            match_key = name_noext

        if match_key is None:
            print(f"[!] WARNING: no CSV entry for '{name_noext}', skipping.")
            continue

        degraded_path = os.path.join(degraded_root, fname)
        mask_path, label = lookup[match_key]

        # print to console for verification
        print(degraded_path, mask_path, label)

        # write to the output TXT
        out_f.write(f"{degraded_path} {mask_path} {label}\n")

print(f"\n✅ Finished. Wrote list to:\n   {output_list}")


/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded/Abyssinian_101_blur10.png /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/masks/Abyssinian_101.png 0
/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded/Abyssinian_101_blur15.png /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/masks/Abyssinian_101.png 0
/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded/Abyssinian_101_blur5.png /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/masks/Abyssinian_101.png 0
/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded/Abyssinian_101_brightGaussian_05.png /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/masks/Abyssinian_101.png 0
/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassification

/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded/British_Shorthair_56_brightGaussian_15.png /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/masks/British_Shorthair_56.png 9
/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded/British_Shorthair_56_brightGlobal.png /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/masks/British_Shorthair_56.png 9
/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded/British_Shorthair_56_obstructed.png /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/masks/British_Shorthair_56.png 9
/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded/British_Shorthair_56_orig.png /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/masks/British_Shorthair_56.png 9
/DATASERVER/M

/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded/Siamese_143_brightGaussian_15.png /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/masks/Siamese_143.png 32
/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded/Siamese_143_brightGlobal.png /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/masks/Siamese_143.png 32
/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded/Siamese_143_obstructed.png /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/masks/Siamese_143.png 32
/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded/Siamese_143_orig.png /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/masks/Siamese_143.png 32
/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxo

/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded/beagle_55_orig.png /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/masks/beagle_55.png 4
/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded/beagle_58_blur10.png /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/masks/beagle_58.png 4
/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded/beagle_58_blur15.png /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/masks/beagle_58.png 4
/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded/beagle_58_blur5.png /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/masks/beagle_58.png 4
/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded/beagle_58_brightGaussian

/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded/great_pyrenees_65_blur5.png /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/masks/great_pyrenees_65.png 15
/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded/great_pyrenees_65_brightGaussian_05.png /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/masks/great_pyrenees_65.png 15
/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded/great_pyrenees_65_brightGaussian_10.png /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/masks/great_pyrenees_65.png 15
/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded/great_pyrenees_65_brightGaussian_15.png /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/masks/great_pyrenees_65.png 15
/DATASERVER/MIC/GENE

/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded/newfoundland_14_orig.png /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/masks/newfoundland_14.png 22
/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded/newfoundland_156_blur10.png /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/masks/newfoundland_156.png 22
/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded/newfoundland_156_blur15.png /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/masks/newfoundland_156.png 22
/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/oxorf_artif_degraded/newfoundland_156_blur5.png /DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassificationmi/data/masks/newfoundland_156.png 22
/DATASERVER/MIC/GENERAL/STUDENTS/amartic/mscthesis/polypclassific

IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)

